In [0]:
%sql
USE CATALOG `retail-sales`;
USE SCHEMA 02_silver;

In [0]:
%sql
drop table if exists `retail-sales`.02_silver.dq_validation_log;

CREATE TABLE `retail-sales`.02_silver.dq_validation_log (
  table_name   STRING,
  check_name   STRING,
  issue_count  BIGINT,
  sample_ids   STRING,
  checked_at   TIMESTAMP
) USING DELTA;

In [0]:
%sql
-- Null check: customers
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'customers_raw', 'null_in_required_columns', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(CustomerID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.customers_raw
WHERE CustomerID IS NULL OR CustomerName IS NULL OR Email IS NULL;

-- Duplicate CustomerID
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'customers_raw', 'duplicate_customer_id',
  COUNT(*) - COUNT(DISTINCT CustomerID),
  'multiple CustomerIDs repeated',
  current_timestamp()
FROM `retail-sales`.01_bronze.customers_raw;

-- Duplicate TransactionID
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'sales_raw', 'duplicate_transaction_id',
  COUNT(*) - COUNT(DISTINCT TransactionID),
  'TransactionID 2481 appears 11 times',
  current_timestamp()
FROM `retail-sales`.01_bronze.sales_raw;

-- Orphan CustomerIDs
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'sales_raw', 'orphan_customer_id', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(s.CustomerID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.sales_raw s
LEFT JOIN `retail-sales`.01_bronze.customers_raw c ON s.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL;

-- Zero Quantity
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'sales_raw', 'quantity_is_zero', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(TransactionID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.sales_raw
WHERE Quantity = 0;

-- Invalid TransactionID
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'sales_raw', 'invalid_transaction_id', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(TransactionID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.sales_raw
WHERE TransactionID > 10000;

-- Zero UnitPrice
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'products_raw', 'unit_price_is_zero', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(ProductID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.products_raw
WHERE UnitPrice = 0;

-- Missing Region
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'stores_raw', 'region_null_or_empty', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(StoreID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.stores_raw
WHERE Region IS NULL OR TRIM(Region) = '';

-- View DQ report
SELECT * FROM `retail-sales`.02_silver.dq_validation_log
ORDER BY table_name, check_name;

In [0]:
%sql
-- DimCustomer (SCD Type 2)


CREATE TABLE IF NOT EXISTS `retail-sales`.02_silver.DimCustomer (
  CustomerSK    BIGINT GENERATED ALWAYS AS IDENTITY,
  CustomerID    INT,
  CustomerName  STRING,
  Email         STRING,
  City          STRING,
  Address       STRING,
  StartDate     DATE,
  EndDate       DATE,
  IsActive      INT
) USING DELTA LOCATION 's3://retail-sales-data-wh/processed/DimCustomer/';

-- DimProduct
CREATE TABLE IF NOT EXISTS `retail-sales`.02_silver.DimProduct (
  ProductSK     BIGINT GENERATED ALWAYS AS IDENTITY,
  ProductID     INT,
  ProductName   STRING,
  Category      STRING,
  UnitPrice     DECIMAL(10,2),
  EffectiveDate DATE
) USING DELTA LOCATION 's3://retail-sales-data-wh/processed/DimProduct/';

-- DimStore
CREATE TABLE IF NOT EXISTS `retail-sales`.02_silver.DimStore (
  StoreSK       BIGINT GENERATED ALWAYS AS IDENTITY,
  StoreID       INT,
  StoreName     STRING,
  Region        STRING
) USING DELTA LOCATION 's3://retail-sales-data-wh/processed/DimStore/';

In [0]:
%sql

TRUNCATE TABLE `retail-sales`.02_silver.DimCustomer;

TRUNCATE TABLE `retail-sales`.02_silver.DimProduct;

TRUNCATE TABLE `retail-sales`.02_silver.DimStore;

In [0]:
%sql
INSERT INTO `retail-sales`.02_silver.DimProduct (
  ProductID, ProductName, Category, UnitPrice, EffectiveDate
)
SELECT
  ProductID,
  TRIM(ProductName)             AS ProductName,
  TRIM(Category)                AS Category,
  UnitPrice,
  CAST(current_date() AS DATE)  AS EffectiveDate
FROM `retail-sales`.01_bronze.products_raw
WHERE UnitPrice > 0;

In [0]:
%sql
INSERT INTO `retail-sales`.02_silver.DimStore (
  StoreID, StoreName, Region
)
SELECT
  StoreID,
  INITCAP(TRIM(StoreName)) AS StoreName,
  COALESCE(TRIM(Region) , 'Unknown')   AS Region
FROM `retail-sales`.01_bronze.stores_raw;

In [0]:
%sql
INSERT INTO `retail-sales`.02_silver.DimCustomer (
  CustomerID, CustomerName, Email, City, Address,
  StartDate, EndDate, IsActive
)
SELECT
  CustomerID,
  INITCAP(TRIM(CustomerName))  AS CustomerName,
  LOWER(TRIM(Email))           AS Email,
  TRIM(City)                   AS City,
  TRIM(Address)                AS Address,
  CAST(current_date() AS DATE) AS StartDate,
  CAST('9999-12-31'   AS DATE) AS EndDate,
  1                            AS IsActive
FROM (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY CustomerID ORDER BY LastUpdated DESC
    ) AS rn
  FROM `retail-sales`.01_bronze.customers_raw
  WHERE CustomerID IS NOT NULL
    AND CustomerName IS NOT NULL
    AND Email IS NOT NULL
) t
WHERE rn = 1;

In [0]:
%sql
SELECT 'DimCustomer' AS table_name, COUNT(*) AS total_rows,
  SUM(CASE WHEN IsActive = 1 THEN 1 ELSE 0 END) AS active_records,
  SUM(CASE WHEN IsActive = 0 THEN 1 ELSE 0 END) AS expired_records
FROM `retail-sales`.02_silver.DimCustomer
UNION ALL
SELECT 'DimProduct', COUNT(*), NULL, NULL
FROM `retail-sales`.02_silver.DimProduct
UNION ALL
SELECT 'DimStore', COUNT(*), NULL, NULL
FROM `retail-sales`.02_silver.DimStore;

In [0]:
%sql

-- =====================================================
-- FULL LOAD VALIDATIONS
-- =====================================================



-- =====================================================
-- VALIDATION 1:
-- Validate total customer count loaded
-- Expected:
-- Around 490 records after removing duplicates/nulls
-- =====================================================

SELECT
    COUNT(*) AS total_customer_records

FROM `retail-sales`.02_silver.DimCustomer;



-- =====================================================
-- VALIDATION 2:
-- Ensure ALL records are ACTIVE during initial load
-- Expected Result: 0 rows
-- =====================================================

SELECT
    *

FROM `retail-sales`.02_silver.DimCustomer

WHERE IsActive != 1;



-- =====================================================
-- VALIDATION 3:
-- Ensure all active rows have EndDate = 9999-12-31
-- Expected Result: 0 rows
-- =====================================================

SELECT
    *

FROM `retail-sales`.02_silver.DimCustomer

WHERE IsActive = 1
AND EndDate != '9999-12-31';



-- =====================================================
-- VALIDATION 4:
-- Ensure only ONE record exists per CustomerID
-- Expected Result: 0 rows
-- =====================================================

SELECT
    CustomerID,
    COUNT(*) AS customer_count

FROM `retail-sales`.02_silver.DimCustomer

GROUP BY CustomerID

HAVING COUNT(*) > 1;



-- =====================================================
-- VALIDATION 5:
-- Validate no NULL critical fields
-- Expected Result: 0 rows
-- =====================================================

SELECT
    *

FROM `retail-sales`.02_silver.DimCustomer

WHERE CustomerID IS NULL
   OR CustomerName IS NULL
   OR Email IS NULL;



-- =====================================================
-- VALIDATION 6:
-- Validate CustomerName proper casing
-- =====================================================

SELECT
    CustomerID,
    CustomerName

FROM `retail-sales`.02_silver.DimCustomer

LIMIT 20;



-- =====================================================
-- VALIDATION 7:
-- Validate Email lowercase transformation
-- =====================================================

SELECT
    CustomerID,
    Email

FROM `retail-sales`.02_silver.DimCustomer

WHERE Email != LOWER(Email);



-- =====================================================
-- VALIDATION 8:
-- Validate City and Address trimming
-- Expected Result: 0 rows
-- =====================================================

SELECT
    CustomerID,
    City,
    Address

FROM `retail-sales`.02_silver.DimCustomer

WHERE City    != TRIM(City)
   OR Address != TRIM(Address);



-- =====================================================
-- VALIDATION 9:
-- Validate StartDate initialization
-- Expected:
-- StartDate should be today's load date
-- =====================================================

SELECT
    DISTINCT StartDate

FROM `retail-sales`.02_silver.DimCustomer;



-- =====================================================
-- VALIDATION 10:
-- Validate surrogate key uniqueness
-- Expected Result: 0 rows
-- =====================================================

SELECT
    CustomerSK,
    COUNT(*) AS duplicate_count

FROM `retail-sales`.02_silver.DimCustomer

GROUP BY CustomerSK

HAVING COUNT(*) > 1;



-- =====================================================
-- VALIDATION 11:
-- Final Full Load Summary
-- =====================================================

SELECT

    COUNT(*) AS total_records,

    SUM(
        CASE
            WHEN IsActive = 1 THEN 1
            ELSE 0
        END
    ) AS active_records,

    SUM(
        CASE
            WHEN IsActive = 0 THEN 1
            ELSE 0
        END
    ) AS inactive_records

FROM `retail-sales`.02_silver.DimCustomer;

In [0]:
%sql

-- =====================================================
-- DIMPRODUCT VALIDATIONS
-- =====================================================



-- =====================================================
-- VALIDATION 1:
-- Validate total records loaded
-- =====================================================

SELECT
    COUNT(*) AS total_product_records

FROM `retail-sales`.02_silver.DimProduct;



-- =====================================================
-- VALIDATION 2:
-- Validate ProductSK uniqueness
-- Expected Result: 0 rows
-- =====================================================

SELECT
    ProductSK,
    COUNT(*) AS duplicate_count

FROM `retail-sales`.02_silver.DimProduct

GROUP BY ProductSK

HAVING COUNT(*) > 1;



-- =====================================================
-- VALIDATION 3:
-- Validate ProductID uniqueness
-- Expected Result: 0 rows
-- =====================================================

SELECT
    ProductID,
    COUNT(*) AS duplicate_count

FROM `retail-sales`.02_silver.DimProduct

GROUP BY ProductID

HAVING COUNT(*) > 1;



-- =====================================================
-- VALIDATION 4:
-- Validate no NULL critical columns
-- Expected Result: 0 rows
-- =====================================================

SELECT
    *

FROM `retail-sales`.02_silver.DimProduct

WHERE ProductID IS NULL
   OR ProductName IS NULL
   OR Category IS NULL
   OR UnitPrice IS NULL;



-- =====================================================
-- VALIDATION 5:
-- Validate ProductName trimming
-- Expected Result: 0 rows
-- =====================================================

SELECT
    ProductID,
    ProductName

FROM `retail-sales`.02_silver.DimProduct

WHERE ProductName != TRIM(ProductName);



-- =====================================================
-- VALIDATION 6:
-- Validate Category trimming
-- Expected Result: 0 rows
-- =====================================================

SELECT
    ProductID,
    Category

FROM `retail-sales`.02_silver.DimProduct

WHERE Category != TRIM(Category);



-- =====================================================
-- VALIDATION 7:
-- Validate UnitPrice > 0
-- Expected Result: 0 rows
-- =====================================================

SELECT
    *

FROM `retail-sales`.02_silver.DimProduct

WHERE UnitPrice <= 0;



-- =====================================================
-- VALIDATION 8:
-- Validate EffectiveDate populated
-- Expected Result: 0 rows
-- =====================================================

SELECT
    *

FROM `retail-sales`.02_silver.DimProduct

WHERE EffectiveDate IS NULL;



-- =====================================================
-- VALIDATION 9:
-- Validate EffectiveDate consistency
-- =====================================================

SELECT
    DISTINCT EffectiveDate

FROM `retail-sales`.02_silver.DimProduct;



-- =====================================================
-- VALIDATION 10:
-- Sample transformed product data
-- =====================================================

SELECT
    ProductSK,
    ProductID,
    ProductName,
    Category,
    UnitPrice,
    EffectiveDate

FROM `retail-sales`.02_silver.DimProduct

LIMIT 20;



-- =====================================================
-- VALIDATION 11:
-- Final DimProduct summary
-- =====================================================

SELECT

    COUNT(*) AS total_products,

    MIN(UnitPrice) AS minimum_price,

    MAX(UnitPrice) AS maximum_price,

    AVG(UnitPrice) AS average_price

FROM `retail-sales`.02_silver.DimProduct;